In [0]:
%pip install python-dotenv

In [0]:
import os
from dotenv import load_dotenv

load_dotenv()

In [0]:
# --- 1. STORAGE CONFIGURATION ---
storage_account_name = "adlsgen2detraining2026"
container_name = "crypto-data-ronit"
storage_account_key = os.getenv("STORAGE_ACCOUNT_KEY")

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)

# --- 2. PATH DEFINITIONS ---
bronze_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze/raw_market_data"
silver_base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver"

# Specific Silver Table Paths
silver_quotes_path = f"{silver_base_path}/crypto_quotes"
silver_global_path = f"{silver_base_path}/global_metrics"
silver_historical_path = f"{silver_base_path}/historical_prices"

# Load Bronze Data
bronze_df = spark.read.format("delta").load(bronze_path)
print(f"✅ Bronze data loaded. Total rows: {bronze_df.count()}")

In [0]:
from pyspark.sql.functions import col, from_json, explode, get_json_object, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, ArrayType

# --- A. PROCESS REAL-TIME QUOTES (From Listings Endpoint) ---
# Note: payload is an array of coins
cmc_listings_schema = ArrayType(StructType([
    StructField("id", StringType()),
    StructField("name", StringType()),
    StructField("symbol", StringType()),
    StructField("quote", StructType([
        StructField("USD", StructType([
            StructField("price", DoubleType()),
            StructField("volume_24h", DoubleType()),
            StructField("market_cap", DoubleType())
        ]))
    ]))
]))

silver_quotes = bronze_df.filter(col("endpoint_type") == "listings") \
    .withColumn("parsed_payload", from_json(get_json_object(col("raw_content"), "$.payload"), cmc_listings_schema)) \
    .withColumn("coin", explode(col("parsed_payload"))) \
    .select(
        col("coin.id").alias("coin_id"),
        col("coin.symbol"),
        col("coin.name"),
        col("coin.quote.USD.price").alias("price_usd"),
        col("coin.quote.USD.volume_24h").alias("volume_24h"),
        col("coin.quote.USD.market_cap").alias("market_cap"),
        col("event_timestamp"),
        current_timestamp().alias("silver_processing_timestamp")
    )

# --- B. PROCESS GLOBAL METRICS ---
# Note: payload is a single object, no explode needed
silver_global = bronze_df.filter(col("endpoint_type") == "global") \
    .select(
        get_json_object(col("raw_content"), "$.payload.quote.USD.total_market_cap").cast("double").alias("total_market_cap"),
        get_json_object(col("raw_content"), "$.payload.quote.USD.total_volume_24h").cast("double").alias("total_volume_24h"),
        get_json_object(col("raw_content"), "$.payload.btc_dominance").cast("double").alias("btc_dominance"),
        col("event_timestamp")
    )

In [0]:
# --- C. PROCESS HISTORICAL BACKFILL ---
# CoinGecko data structure: [ [timestamp, value], [timestamp, value] ]
coingecko_schema = StructType([
    StructField("prices", ArrayType(ArrayType(DoubleType()))),
    StructField("market_caps", ArrayType(ArrayType(DoubleType()))),
    StructField("total_volumes", ArrayType(ArrayType(DoubleType())))
])

historical_df = bronze_df.filter(col("endpoint_type") == "historical_backfill")

# We use from_json to turn the payload into structured columns, then explode the arrays
silver_historical = historical_df.withColumn("payload_json", from_json(get_json_object(col("raw_content"), "$.payload"), coingecko_schema)) \
    .withColumn("exploded_prices", explode(col("payload_json.prices"))) \
    .select(
        get_json_object(col("raw_content"), "$.coin_id").alias("coin_id"),
        (col("exploded_prices")[0] / 1000).cast("timestamp").alias("price_timestamp"), # ms to seconds
        col("exploded_prices")[1].alias("price_usd"),
        col("source")
    )

In [0]:
# Save as Delta tables with MergeSchema enabled in case API adds fields later
print("Writing Silver tables...")

silver_quotes.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(silver_quotes_path)
silver_global.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(silver_global_path)
silver_historical.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(silver_historical_path)

print("✅ Silver Layer Processing Complete!")

In [0]:
print("Silver Quotes Sample:")
display(spark.read.format("delta").load(silver_quotes_path).limit(5))

print("Silver Historical Sample:")
display(spark.read.format("delta").load(silver_historical_path).limit(5))

print("Silver Global Metrics Sample:")
display(spark.read.format("delta").load(silver_global_path).limit(5))